In [1]:
import pandas as pd
import numpy as np
import random
from typing import List, Dict, Tuple
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import json
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import os
from datetime import datetime

# Import our bias analysis framework
from LLM_debias import LLMPositionBiasAnalyzer

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

# Configuration
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Libraries imported successfully")


✅ Libraries imported successfully


In [2]:
# Define paths - directly use files in data/news folder
DATA_DIR = "data/news/"
NEWS_FILE = os.path.join(DATA_DIR, "news.tsv")
BEHAVIORS_FILE = os.path.join(DATA_DIR, "behaviors.tsv")

# Check if data files exist
print(f"📁 Data directory: {DATA_DIR}")
print(f"📰 News file exists: {os.path.exists(NEWS_FILE)}")
print(f"👤 Behaviors file exists: {os.path.exists(BEHAVIORS_FILE)}")

# List available files in data directory
if os.path.exists(DATA_DIR):
    data_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.tsv')]
    print(f"\n📄 TSV files in data directory:")
    for file in data_files:
        file_path = os.path.join(DATA_DIR, file)
        if os.path.isfile(file_path):
            size_mb = os.path.getsize(file_path) / (1024 * 1024)
            print(f"  {file}: {size_mb:.2f} MB")
else:
    print(f"\n⚠️ Data directory {DATA_DIR} not found.")


📁 Data directory: data/news/
📰 News file exists: True
👤 Behaviors file exists: True

📄 TSV files in data directory:
  behaviors.tsv: 87.76 MB
  news.tsv: 39.29 MB


In [3]:
# # Load news.tsv
# print(f"📰 Loading news data from: {NEWS_FILE}")

# # Define column names for news.tsv
# news_columns = [
#     'NewsID', 'Category', 'SubCategory', 'Title', 
#     'Abstract', 'URL', 'TitleEntities', 'AbstractEntities'
# ]

# # Load news data
# news_df = pd.read_csv(NEWS_FILE, sep='\t', header=None, names=news_columns)

# print(f"📊 News dataset shape: {news_df.shape}")
# print(f"📊 Unique news articles: {news_df['NewsID'].nunique()}")
# print(f"📊 Categories: {news_df['Category'].nunique()}")
# print(f"📊 SubCategories: {news_df['SubCategory'].nunique()}")

# # Display sample news data
# print("\n📰 Sample news data:")
# print(news_df.head())


In [4]:
# # Explore news categories
# print("📊 News Categories Distribution:")
# category_counts = news_df['Category'].value_counts()
# print(category_counts.head(10))

# # Check for missing titles
# missing_titles = news_df['Title'].isna().sum()
# print(f"\n⚠️ News articles with missing titles: {missing_titles}")

# # Show sample titles
# print("\n📰 Sample news titles:")
# for i, title in enumerate(news_df['Title'].head(10)):
#     print(f"  {i+1}. {title}")


In [5]:
# Load behaviors.tsv
print(f"👤 Loading behaviors data from: {BEHAVIORS_FILE}")

# Define column names for behaviors.tsv
behaviors_columns = [
    'ImpressionID', 'UserID', 'Time', 'History', 'Impressions'
]

# Load behaviors data
behaviors_df = pd.read_csv(BEHAVIORS_FILE, sep='\t', header=None, names=behaviors_columns)

print(f"📊 Behaviors dataset shape: {behaviors_df.shape}")
print(f"📊 Unique users: {behaviors_df['UserID'].nunique()}")
print(f"📊 Unique impressions: {behaviors_df['ImpressionID'].nunique()}")

# Display sample behaviors data
print("\n👤 Sample behaviors data:")
print(behaviors_df.head())


👤 Loading behaviors data from: data/news/behaviors.tsv
📊 Behaviors dataset shape: (156965, 5)
📊 Unique users: 50000
📊 Unique impressions: 156965

👤 Sample behaviors data:
   ImpressionID  UserID                   Time  \
0             1  U13740  11/11/2019 9:05:58 AM   
1             2  U91836  11/12/2019 6:11:30 PM   
2             3  U73700  11/14/2019 7:01:48 AM   
3             4  U34670  11/11/2019 5:28:05 AM   
4             5   U8125  11/12/2019 4:11:21 PM   

                                             History  \
0  N55189 N42782 N34694 N45794 N18445 N63302 N104...   
1  N31739 N6072 N63045 N23979 N35656 N43353 N8129...   
2  N10732 N25792 N7563 N21087 N41087 N5445 N60384...   
3  N45729 N2203 N871 N53880 N41375 N43142 N33013 ...   
4                        N10078 N56514 N14904 N33740   

                                         Impressions  
0                                  N55689-1 N35729-0  
1  N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...  
2  N50014-0 N23877-0 N3538

In [6]:
# filtered_df = behaviors_df[behaviors_df['UserID'] == 'U89249']['Impressions']

# with pd.option_context('display.max_colwidth', None):
#     print(filtered_df)

In [7]:
# # Explore user behaviors
# print("📊 User Behavior Statistics:")

# # Check for missing histories
# missing_history = behaviors_df['History'].isna().sum()
# print(f"Users with no history: {missing_history}")

# # Analyze history lengths
# history_lengths = []
# for idx, history in enumerate(behaviors_df['History']):
#     if pd.notna(history):
#         history_length = len(history.split())
#         history_lengths.append(history_length)
#     else:
#         history_lengths.append(0)

# history_lengths = np.array(history_lengths)
# print(f"\n📊 History Length Statistics:")
# print(f"  Mean: {history_lengths.mean():.2f}")
# print(f"  Median: {np.median(history_lengths):.2f}")
# print(f"  Min: {history_lengths.min()}")
# print(f"  Max: {history_lengths.max()}")
# print(f"  Users with ≥5 history items: {sum(history_lengths >= 5)}")

# # Analyze impression lengths
# impression_lengths = []
# for impressions in behaviors_df['Impressions']:
#     if pd.notna(impressions):
#         impression_length = len(impressions.split())
#         impression_lengths.append(impression_length)
#     else:
#         impression_lengths.append(0)

# impression_lengths = np.array(impression_lengths)
# print(f"\n📊 Impression Length Statistics:")
# print(f"  Mean: {impression_lengths.mean():.2f}")
# print(f"  Median: {np.median(impression_lengths):.2f}")
# print(f"  Min: {impression_lengths.min()}")
# print(f"  Max: {impression_lengths.max()}")


In [8]:
# def process_mind_dataset(behaviors_df, news_df, min_history_length=5, max_candidates=100):
#     """
#     Process MIND dataset for LLM bias analysis.
    
#     Args:
#         behaviors_df: DataFrame with user behaviors
#         news_df: DataFrame with news information
#         min_history_length: Minimum required history length
#         max_candidates: Maximum number of candidates to generate
    
#     Returns:
#         DataFrame suitable for LLMPositionBiasAnalyzer
#     """
#     processed_data = []
    
#     # Create news ID to title mapping with category and subcategory
#     news_id_to_title = {}
#     for _, row in news_df.iterrows():
#         news_id = row['NewsID']
#         title = row['Title']
#         category = row['Category']
#         subcategory = row['SubCategory']
        
#         # Format: {News Title}[Category:{Category},Subcategory:{Subcategory}]
#         formatted_title = f"{title}[Category:{category},Subcategory:{subcategory}]"
#         news_id_to_title[news_id] = formatted_title
    
#     # Get all available news IDs for negative sampling
#     all_news_ids = set(news_df['NewsID'].tolist())
    
#     print(f"🔄 Processing {len(behaviors_df)} user behaviors...")
    
#     processed_users = set()  # Track processed users to avoid duplicates
    
#     for idx, row in tqdm(behaviors_df.iterrows(), total=len(behaviors_df), desc="Processing behaviors"):
#         user_id = row['UserID']
        
#         # Skip if user already processed (handle duplicates)
#         if user_id in processed_users:
#             continue
        
#         history = row['History']
#         impressions = row['Impressions']
        
#         # Skip if no history or impressions
#         if pd.isna(history) or pd.isna(impressions):
#             continue
        
#         # Parse history
#         history_news_ids = history.split()
        
#         # Skip if insufficient history
#         if len(history_news_ids) < min_history_length:
#             continue
        
#         # Parse impressions
#         impression_items = impressions.split()
#         clicked_news = []
#         unclicked_news = []
        
#         for item in impression_items:
#             if '-' in item:
#                 news_id, click = item.split('-')
#                 if click == '1':
#                     clicked_news.append(news_id)
#                 else:
#                     unclicked_news.append(news_id)
        
#         # Skip if no clicked news
#         if not clicked_news:
#             continue
        
#         # Take one clicked news as target (randomly if multiple)
#         target_news_id = random.choice(clicked_news)
        
#         # Get negative candidates from unclicked news
#         negative_candidates = unclicked_news.copy()
        
#         # Add random negatives if needed
#         user_seen_news = set(history_news_ids + clicked_news + unclicked_news)
#         available_negatives = list(all_news_ids - user_seen_news)
        
#         # if len(negative_candidates) < max_candidates - 1:
#         #     additional_negatives = random.sample(
#         #         available_negatives, 
#         #         min(max_candidates - 1 - len(negative_candidates), len(available_negatives))
#         #     )
#         #     negative_candidates.extend(additional_negatives)
        
#         # Limit to max_candidates - 1 (leave room for target)
#         negative_candidates = negative_candidates[:max_candidates - 1]
        
#         # Convert news IDs to titles with category and subcategory
#         def get_title(news_id):
#             return news_id_to_title.get(news_id, f"Unknown_{news_id}[Category:Unknown,Subcategory:Unknown]")
        
#         # Get last 5 history items as titles with categories
#         history_titles = [get_title(nid) for nid in history_news_ids[-5:]]
        
#         # Get target title with category
#         target_title = get_title(target_news_id)
        
#         # Create entries for history items
#         current_time = idx  # Use index as timestamp
        
#         # Add history items
#         for i, title in enumerate(history_titles):
#             processed_data.append({
#                 'UserID': user_id,
#                 'Title': title,
#                 'Timestamp': current_time - len(history_titles) + i
#             })
        
#         # Add target item
#         processed_data.append({
#             'UserID': user_id,
#             'Title': target_title,
#             'Timestamp': current_time
#         })
        
#         # Add negative candidates (with earlier timestamps to ensure they're not selected as targets)
#         for i, neg_news_id in enumerate(negative_candidates):
#             neg_title = get_title(neg_news_id)
#             processed_data.append({
#                 'UserID': f"{user_id}_neg_{i}",  # Different user ID for negatives
#                 'Title': neg_title,
#                 'Timestamp': current_time - 1000 - i  # Much earlier timestamp
#             })
        
#         processed_users.add(user_id)
    
#     print(f"✅ Processed {len(processed_users)} unique users")
#     print(f"✅ Generated {len(processed_data)} total data points")
    
#     return pd.DataFrame(processed_data)

# # Process the dataset
# processed_df = process_mind_dataset(behaviors_df, news_df, min_history_length=5, max_candidates=20)

# print(f"\n📊 Processed Dataset Statistics:")
# print(f"  Total rows: {len(processed_df)}")
# print(f"  Unique users: {processed_df['UserID'].nunique()}")
# print(f"  Unique news titles: {processed_df['Title'].nunique()}")
# print(f"  Date range: {processed_df['Timestamp'].min()} to {processed_df['Timestamp'].max()}")

In [9]:
# output_path = './data/news/processed_df3.csv'
# processed_df.to_csv(output_path, index=False)

In [10]:
processed_df = pd.read_csv('./data/news/processed_df3.csv')

In [11]:
# filtered_df = processed_df[processed_df['UserID'].astype(str).str.startswith('U89249')]['Title'].tolist()

# # Print the result
# print(filtered_df)

In [12]:
# # Display sample processed data
# print("📊 Sample Processed Data:")
# print(processed_df.head(20))

# # Check user distribution
# user_counts = processed_df['UserID'].value_counts()
# print(f"\n📊 User Interaction Distribution:")
# print(f"  Users with 6+ interactions: {sum(user_counts >= 6)}")
# print(f"  Users with 10+ interactions: {sum(user_counts >= 10)}")
# print(f"  Users with 20+ interactions: {sum(user_counts >= 20)}")

# print(f"\n📊 Sample User Interaction Counts:")
# print(user_counts.head(10))


In [13]:
# Configuration for bias analysis
MODEL_NAME = 'gpt-3.5-turbo'
BACKEND = 'openai'
DATA_NAME = 'news'
API_TIER = 'tier_2'  # Adjust based on your API limits

# Analysis parameters
NUM_BIAS_USERS = 5      # Users for bias detection
NUM_EVAL_USERS = 200      # Users for evaluation
NUM_SHUFFLES_BIAS = 50   # Number of shuffles for bias detection
LIST_SIZE = 20          # Size of candidate lists
NUM_TRIALS = 20          # Number of randomization trials

print(f"🔧 Configuration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Backend: {BACKEND}")
print(f"  Dataset: {DATA_NAME}")
print(f"  API Tier: {API_TIER}")
print(f"  Bias Users: {NUM_BIAS_USERS}")
print(f"  Eval Users: {NUM_EVAL_USERS}")
print(f"  Shuffles: {NUM_SHUFFLES_BIAS}")
print(f"  List Size: {LIST_SIZE}")
print(f"  Trials: {NUM_TRIALS}")

# Initialize the analyzer
print(f"\n🚀 Initializing LLM Bias Analyzer...")
analyzer = LLMPositionBiasAnalyzer(
    data=processed_df,
    data_name=DATA_NAME,
    model=MODEL_NAME,
    backend=BACKEND,
    num_bias_users=NUM_BIAS_USERS,
    num_eval_users=NUM_EVAL_USERS,
    num_shuffles_bias=NUM_SHUFFLES_BIAS,
    list_size=LIST_SIZE,
    api_tier=API_TIER
)

print(f"✅ Analyzer initialized successfully!")


🔧 Configuration:
  Model: gpt-3.5-turbo
  Backend: openai
  Dataset: news
  API Tier: tier_2
  Bias Users: 5
  Eval Users: 200
  Shuffles: 50
  List Size: 20
  Trials: 20

🚀 Initializing LLM Bias Analyzer...
📊 User filtering results (News dataset - no minimum interaction filter):
  Total users in dataset: 607268
  Real users (excluding negatives): 40331
  Using all real users for analysis
✅ Selected 5 bias users and 200 evaluation users
   All selected users have ≥6 items for reliable evaluation
Initialized LLM Bias Analyzer:
  Model: gpt-3.5-turbo
  Backend: openai
  API Tier: tier_2
  Rate Limits: 5000 RPM, 2000000 TPM
  Max Workers: 25
  Batch Size: 50
  Request Delay: 0.030s
✅ Analyzer initialized successfully!


In [14]:
# bias_analysis =  analyzer.compute_bias_analysis(5,None,False,None,20)
# print(bias_analysis)

In [17]:
# {'avg_primacy': 1.072, 'avg_recency': 0.02, 'avg_middle': 0.908}
# prebias_gpt35_news = {'avg_primacy': 1.072,
#  'avg_recency': 0.02,
 # 'avg_middle': 0.908}

prebias_gpt35_news = {'avg_primacy': 0.44400000000000006,
                      'avg_recency': 0.6039999999999999,
                      'avg_middle': 0.9520000000000002}


In [18]:
# Main debiasing experiment with list size 20
print("\n🔧 COMPLETE DEBIASING EXPERIMENT - NEWS")
print("=" * 50)

num_candidates = 20    # Number of candidates per evaluation
num_trials = 20        # Number of randomization trials per user
batch_size =  20     # Batch size for processing

print(f"🎯 Candidates per evaluation: {num_candidates}")
print(f"🔄 Trials per user: {num_trials}")
print(f"📦 Batch size: {batch_size}")

# Run the complete evaluation pipeline
print("\n🚀 RUNNING COMPLETE EVALUATION PIPELINE...")
print("This will:")
print("1. 📊 Select separate users for bias detection vs evaluation")
print("2. 🔍 Run bias detection on bias detection users")  
print("3. ⚖️ Calculate propensity scores from detected bias")
print("4. 📈 Evaluate on evaluation users using calculated propensity scores")
print("5. 💾 Save all raw data for future reanalysis")

# Uncomment and run the evaluation below
results = analyzer.evaluate_our_method_batched(
    num_candidates=num_candidates,
    num_trials=num_trials,
    aggregation_method="mean",
    batch_size=batch_size,
    use_parallel=True,
    precalculated_bias=prebias_gpt35_news,
    checkpoint_file="evaluation_checkpoint_news9.json"
)


🔧 COMPLETE DEBIASING EXPERIMENT - NEWS
🎯 Candidates per evaluation: 20
🔄 Trials per user: 20
📦 Batch size: 20

🚀 RUNNING COMPLETE EVALUATION PIPELINE...
This will:
1. 📊 Select separate users for bias detection vs evaluation
2. 🔍 Run bias detection on bias detection users
3. ⚖️ Calculate propensity scores from detected bias
4. 📈 Evaluate on evaluation users using calculated propensity scores
5. 💾 Save all raw data for future reanalysis
📁 Checkpoint file: evaluation_checkpoint_news9.json
📂 Resuming from checkpoint: 0 users already completed
API Tier: tier_2 (RPM: 5000, TPM: 2000000)
Max workers - Bias: 25, Trials: 12, Users: 3
📊 Using bias analysis from checkpoint
👥 Total users: 200, Completed: 0, Remaining: 200

🔄 Processing batch 1/10 (20 users)
Evaluating 20 users in parallel with max_workers=3...

Running 20 randomization trials (with raw data preservation)...
Running 20 randomization trials (with raw data preservation)...
Executing 20 trials in parallel with max_workers=12...
Rate 




atch 1/1):   0%|                                | 0/20 [00:00<?, ?it/s]



h 1/1):   0%|                                | 0/20 [00:00<?, ?it/s]




/1):   0%|                                | 0/20 [00:00<?, ?it/s]



h 1/1):   5%|█▏                      | 1/20 [00:00<00:17,  1.06it/s]




/1):   5%|█▏                      | 1/20 [00:01<00:21,  1.11s/it]



h 1/1):  10%|██▍                     | 2/20 [00:01<00:08,  2.02it/s]


atch 1/1):   5%|█▏                      | 1/20 [00:01<00:22,  1.20s/it]


atch 1/1):  15%|███▌                    | 3/20 [00:01<00:06,  2.76it/s]



h 1/1):  15%|███▌                    | 3/20 [00:01<00:06,  2.45it/s]


atch 1/1):  20%|████▊                   | 4/20 [00:01<00:04,  3.52it/s]


atch 1/1):  30%|███████▏                | 6/20 [00:01<00:02,  5.87it/s]




/1):  15%|███▌                    | 3/20 [00:01<00:07,  2.17it/s]



h 1/1):  30%|███████▏                | 6/20 [00:01<00:02,  5.49it/s]




/1):  35%|████████▍               | 7/20 [00:01<00:02, 

Completed 20 successful trials out of 20 attempted







/1):  95%|█████████████████████▊ | 19/20 [00:03<00:00,  6.10it/s]


Running 20 randomization trials (with raw data preservation)...
Executing 20 trials in parallel with max_workers=12...
Rate limiting: 12 workers, 0.030s delay, batch size: 50


...

Completed 20 successful trials out of 20 attempted






h 1/1):  25%|██████                  | 5/20 [00:01<00:02,  5.22it/s]



h 1/1):  40%|█████████▌              | 8/20 [00:01<00:01,  8.62it/s]



h 1/1):  55%|████████████▋          | 11/20 [00:01<00:00, 10.74it/s]




/1):   5%|█▏                      | 1/20 [00:01<00:22,  1.17s/it]




/1):  10%|██▍                     | 2/20 [00:01<00:11,  1.63it/s]




/1):  25%|██████                  | 5/20 [00:01<00:03,  4.38it/s]




/1):  40%|█████████▌              | 8/20 [00:01<00:01,  6.78it/s]



h 1/1):  65%|██████████████▉        | 13/20 [00:02<00:01,  4.10it/s]



h 1/1):  75%|█████████████████▎     | 15/20 [00:03<00:00,  5.18it/s]




/1):  50%|███████████▌           | 10/20 [00:02<00:01,  6.88it/s]



h 1/1):  85%|███████████████████▌   | 17/20 [00:03<00:00,  6.37it/s]



Trials (batch 1/1): 100%|███████████████████████| 20/20 [00:03<00:00,  5.57it/s]


Completed 20 successful trials out of 20 attempted







/1):  65%|██████████████▉        | 13/20 [00:02<00:01,  5.26it/s]




/1):  70%|████████████████       | 14/20 [00:03<00:01,  4.80it/s]




/1):  80%|██████████████████▍    | 16/20 [00:03<00:00,  5.48it/s]




/1):  85%|███████████████████▌   | 17/20 [00:03<00:00,  5.79it/s]




Trials (batch 1/1): 100%|███████████████████████| 20/20 [00:03<00:00,  5.33it/s]


Completed 20 successful trials out of 20 attempted
Progress: 20/20 users completed
✅ Completed evaluation of 20 users
✅ Batch 10 completed. Progress: 200/200 users

📊 Computing final metrics from 200 user results...

OUR METHOD EVALUATION RESULTS vs BENCHMARKS

Our Method Results:
  Accuracy:    0.0450 ± 0.2073
  NDCG@1:      0.0450 ± 0.2073
  NDCG@5:      0.1452 ± 0.2709
  NDCG@10:     0.2047 ± 0.2670
  NDCG@20:     0.3433 ± 0.1735
  Number of evaluations: 200

Benchmark Results (Accuracy) - From Paper:
Method          Movie Dataset  
------------------------------
Raw Output      0.2740±0.0593
Bootstrapping   0.2537
STELLA          0.2976
Our Method      0.0450±0.2073

Accuracy Comparison (Movie Dataset):
----------------------------------------
Our Method vs Raw Output:    -0.2290
Our Method vs Bootstrapping: -0.2087
Our Method vs STELLA:        -0.2526

NDCG Analysis:
----------------------------------------
NDCG@1 = Accuracy: 0.0450
NDCG@5:  0.1452 (322.6% of NDCG@1)
NDCG@10: 0.20

In [21]:
# ## Raw LLM Output Accuracy for News Dataset
# This cell computes the accuracy of the raw LLM outputs (before debiasing) from the evaluation checkpoints for the news dataset.

import glob
import json
import numpy as np
import os

def get_latest_news_checkpoint():
    files = glob.glob("evaluation_checkpoint_news*.json")
    if not files:
        raise FileNotFoundError("No news checkpoint files found.")
    # Pick the largest file (most complete)
    files = sorted(files, key=lambda f: os.path.getsize(f), reverse=True)
    return files[2]

checkpoint_file = get_latest_news_checkpoint()
print(f"Using checkpoint: {checkpoint_file}")

with open(checkpoint_file, 'r') as f:
    data = json.load(f)

# Try all possible user result keys
user_result_keys = ['user_results', 'all_user_results', 'recomputed_user_results']
user_results = None
for k in user_result_keys:
    if k in data:
        user_results = data[k]
        break
if user_results is None:
    raise ValueError("No user results found in checkpoint.")

accuracies = []
ndcg5s = []
ndcg10s = []
for user in user_results:
    # Must have raw_llm_data or similar
    raw_trials = user.get('raw_llm_data')
    target = user.get('target_item')
    if not raw_trials or not target:
        continue
    # Use the first trial (raw LLM output, before debiasing)
    first_trial = raw_trials[0]
    llm_reranked_list = first_trial.get('llm_reranked_list')
    if not llm_reranked_list:
        continue
    ranked_titles = [item['title'] for item in llm_reranked_list]
    # Accuracy: is target ranked first?
    acc = 1.0 if ranked_titles and ranked_titles[0] == target else 0.0
    accuracies.append(acc)
    # NDCG@5 and NDCG@10
    def ndcg_at_k(target, ranked, k):
        try:
            idx = ranked.index(target)
        except ValueError:
            return 0.0
        if idx >= k:
            return 0.0
        return 1.0 / np.log2(idx + 2)
    ndcg5s.append(ndcg_at_k(target, ranked_titles, 5))
    ndcg10s.append(ndcg_at_k(target, ranked_titles, 10))

n = len(accuracies)
print(f"\nRaw LLM Output Accuracy (News): {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f} (n={n})")
print(f"Raw LLM Output NDCG@5: {np.mean(ndcg5s):.4f}")
print(f"Raw LLM Output NDCG@10: {np.mean(ndcg10s):.4f}")

Using checkpoint: evaluation_checkpoint_news9.json

Raw LLM Output Accuracy (News): 0.0600 ± 0.2375 (n=200)
Raw LLM Output NDCG@5: 0.1427
Raw LLM Output NDCG@10: 0.2220


In [ ]:
# Test debiasing on a sample user
print("🧪 Testing debiasing on sample user...")

# Create ranking for debiasing
ranked_candidates, user_history = analyzer.create_ranking_for_debiasing(sample_user_id)

if ranked_candidates:
    print(f"\n📊 Original Ranking (Top 10):")
    for i, item in enumerate(ranked_candidates[:10]):
        title = item['title'][:60] + "..." if len(item['title']) > 60 else item['title']
        print(f"  {i+1}. {title} (score: {item.get('llm_score', 0):.3f})")
    
    # Apply debiasing
    debiased_candidates = analyzer.debias_ranking(ranked_candidates, method="inverse_propensity")
    
    print(f"\n📊 Debiased Ranking (Top 10):")
    for i, item in enumerate(debiased_candidates[:10]):
        title = item['title'][:60] + "..." if len(item['title']) > 60 else item['title']
        print(f"  {i+1}. {title} (debiased: {item.get('debiased_score', 0):.3f}, weight: {item.get('propensity_weight', 0):.3f})")
    
    # Compare rankings
    analyzer.compare_rankings(ranked_candidates, debiased_candidates, top_k=15)
    
    print(f"\n👤 User History:")
    for i, hist_item in enumerate(user_history):
        print(f"  {i+1}. {hist_item[:80]}...") if len(hist_item) > 80 else print(f"  {i+1}. {hist_item}")
else:
    print("❌ Could not create ranking for sample user")


In [ ]:
# Test randomization and aggregation
print("🎲 Testing randomization and aggregation...")

if ranked_candidates:
    # Run randomization with fewer trials for testing
    test_trials = 5
    aggregation_results = analyzer.randomize_and_aggregate_scores(
        candidate_list=ranked_candidates,
        user_history=user_history,
        num_trials=test_trials,
        aggregation_method="mean",
        propensity_scores=propensity_scores,
        use_parallel=True,
        max_workers=3
    )
    
    print(f"\n📊 Randomization Results:")
    print(f"  Successful trials: {aggregation_results['successful_trials']}/{aggregation_results['requested_trials']}")
    print(f"  Aggregation method: {aggregation_results['aggregation_method']}")
    
    if 'final_ranking' in aggregation_results:
        final_ranking = aggregation_results['final_ranking']
        print(f"\n📊 Final Aggregated Ranking (Top 10):")
        for i, item in enumerate(final_ranking[:10]):
            title = item['title'][:60] + "..." if len(item['title']) > 60 else item['title']
            agg_score = item.get('aggregated_score', 0)
            debiased_score = item.get('aggregated_debiased_score', 0)
            weight = item.get('avg_propensity_weight', 0)
            print(f"  {i+1}. {title} (agg: {agg_score:.3f}, debiased: {debiased_score:.3f}, weight: {weight:.3f})")
    else:
        print("⚠️ No final ranking available")
else:
    print("❌ Cannot test randomization without ranked candidates")


In [ ]:
# This cell is optional and resource-intensive
# Uncomment to run full evaluation

# Create precalculated bias for efficiency
precalculated_bias = analyzer.create_precalculated_bias_dict(
    primacy=bias_result['avg_primacy'],
    recency=bias_result['avg_recency'],
    middle=bias_result['avg_middle']
)

print(f"📊 Precalculated bias scores:")
for key, value in precalculated_bias.items():
    print(f"  {key}: {value:.3f}")

# Show API recommendations
print(f"\n💡 API Recommendations:")
recommendations = analyzer.get_api_recommendations(your_rpm=3500, your_tpm=1000000)

print(f"\n🚀 To run full evaluation, uncomment the following code:")
print(f"# results = analyzer.evaluate_our_method_batched(")
print(f"#     batch_size=10,")
print(f"#     num_candidates=20,")
print(f"#     num_trials=10,")
print(f"#     precalculated_bias=precalculated_bias,")
print(f"#     checkpoint_file='evaluation_checkpoint_news.json',")
print(f"#     resume_from_checkpoint=True")
print(f"# )")


In [ ]:
# Uncomment to run full evaluation
# WARNING: This will make many API calls and may take significant time

# results = analyzer.evaluate_our_method_batched(
#     batch_size=10,
#     num_candidates=20,
#     num_trials=10,
#     precalculated_bias=precalculated_bias,
#     checkpoint_file='evaluation_checkpoint_news.json',
#     resume_from_checkpoint=True
# )

# print(f"\n🎉 Evaluation Results:")
# if 'our_method_evaluation' in results:
#     our_results = results['our_method_evaluation']
#     print(f"  Accuracy: {our_results['accuracy']['mean']:.4f} ± {our_results['accuracy']['std']:.4f}")
#     print(f"  NDCG@5: {our_results['ndcg_5']['mean']:.4f} ± {our_results['ndcg_5']['std']:.4f}")
#     print(f"  NDCG@10: {our_results['ndcg_10']['mean']:.4f} ± {our_results['ndcg_10']['std']:.4f}")
#     print(f"  Number of evaluations: {our_results['accuracy']['num_evaluations']}")
# else:
#     print(f"  Results: {results}")

print("💡 To run evaluation, uncomment the code above")


In [ ]:
# Save bias analysis results
bias_results = {
    'dataset': 'news_mind',
    'model': MODEL_NAME,
    'backend': BACKEND,
    'api_tier': API_TIER,
    'configuration': {
        'num_bias_users': NUM_BIAS_USERS,
        'num_eval_users': NUM_EVAL_USERS,
        'num_shuffles_bias': NUM_SHUFFLES_BIAS,
        'list_size': LIST_SIZE,
        'num_trials': NUM_TRIALS
    },
    'bias_detection_results': bias_result,
    'bias_scores': bias_scores,
    'propensity_scores': propensity_scores,
    'precalculated_bias': precalculated_bias,
    'dataset_stats': {
        'total_behaviors': len(behaviors_df),
        'total_news': len(news_df),
        'processed_interactions': len(processed_df),
        'unique_users': processed_df['UserID'].nunique(),
        'unique_news': processed_df['Title'].nunique()
    }
}

# Save to JSON file
output_file = f'news_mind_bias_analysis_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
with open(output_file, 'w') as f:
    json.dump(bias_results, f, indent=2, default=str)

print(f"💾 Results saved to: {output_file}")

# Display final summary
print(f"\n📊 FINAL SUMMARY - News MIND Dataset Analysis")
print(f"=" * 50)
print(f"Dataset: {bias_results['dataset']}")
print(f"Model: {bias_results['model']}")
print(f"\n📈 Bias Scores:")
for bias_type, score in bias_scores.items():
    print(f"  {bias_type}: {score:.3f}")
print(f"\n📊 Dataset Statistics:")
for key, value in bias_results['dataset_stats'].items():
    print(f"  {key}: {value:,}")
print(f"\n💡 To run full evaluation:")
print(f"  1. Uncomment the evaluation code above")
print(f"  2. Monitor API usage and costs")
print(f"  3. Results will be saved to checkpoint files")
print(f"\n✅ News MIND dataset analysis setup complete!")
